In [ ]:
import numpy as np
import pandas as pd
from datasets import load_dataset
import random
import warnings

# 경고 메시지 무시 (튜토리얼 환경에서 발생하는 가벼운 경고들을 위해)
warnings.filterwarnings("ignore")

# =================================================================
# ✨ 🛰️ AI 튜토리얼: 위성 사진에서 구름을 지우는 마법 (Cloud Removal in Satellite Imagery)
# =================================================================
# 🌍 이 데이터셋은 'Hermanni/sen12mscr'입니다.
# 📝 한글 제목: 센티넬 12m 구름 제거 데이터셋 (SEN12MS-CR)
# 💡 의미: 지구를 관찰하는 위성(Sentinel)이 촬영한 사진에서 '구름'이나 '어두운 부분'을 제거하고,
#      보다 깨끗한 '지표면 정보'만 남기기 위한 AI 실습에 사용됩니다.
# 🚀 목표: 초보자도 따라 할 수 있게, 데이터를 탐색하고, 어떤 조건에서 AI가 필요했는지 '분석'해보는 실습을 진행합니다!
# =================================================================

# 실습에 사용할 샘플 개수 (너무 많으면 오래 걸리니까 적당히! 😊)
SAMPLE_COUNT = 100
DATASET_NAME = "Hermanni/sen12mscr"

print("=" * 70)
print("✨🛰️ 파이썬 AI 데이터 탐험가 코스에 오신 것을 환영합니다! 🚀")
print("=" * 70)

# 1. 데이터 로드 전략 설정 (스트리밍 vs. 일반 로드)
dataset = None
print(f"[STEP 1] 📚 데이터 로딩 시도: {DATASET_NAME}...")

try:
    # 1-1. 스트리밍 방식으로 로드 시도 (가장 빠르고 메모리 효율적!)
    dataset = load_dataset(DATASET_NAME, split='train', streaming=True)
    print("✅ 성공! 스트리밍 데이터셋(IterableDataset)으로 로드되었습니다. 메모리 걱정 없이 순차적으로 탐험 가능해요!")
    
    # 스트리밍 환경에 최적화된 샘플러를 준비합니다.
    # .take() 패턴은 스트리밍 데이터에서 처음 K개만 가져오는 가장 안전한 방법입니다.
    print(f"   -> 처음 {SAMPLE_COUNT}개 샘플만 가져와서 실습에 활용하겠습니다.")

except Exception as e:
    # 만약 스트리밍이 실패하거나 메모리가 부족하면 일반 로드로 전환합니다.
    print(f"⚠️ 스트리밍 로드 실패 또는 오류 발생 ({type(e).__name__}). 일반 데이터셋(Dataset)으로 전환합니다.")
    try:
        # 1-2. 일반 데이터셋으로 로드 (메모리를 많이 사용할 수 있어요!)
        dataset = load_dataset(DATASET_NAME, split='train', streaming=False)
        print(f"✅ 일반 데이터셋으로 {len(dataset)}개의 샘플을 메모리에 로드했습니다. 이제 탐색을 시작할게요!")
    except Exception as e_fall:
        print(f"❌ 데이터를 로드하는 데 실패했습니다. 에러: {e_fall}")
        exit()


# 2. 샘플 데이터 준비 (K개 샘플을 반복 가능한 형태로 만듭니다.)
if hasattr(dataset, "take"):
    # .take()가 존재하면 스트리밍 데이터셋(IterableDataset)입니다.
    print("\n[STEP 2] ♻️ 샘플 데이터셋 준비 (스트리밍 최적화)")
    # 스트리밍 데이터셋은 list(dataset.take(K)) 패턴을 사용합니다.
    sampled_dataset = list(dataset.take(SAMPLE_COUNT))
    print(f"   -> 총 {len(sampled_dataset)}개의 샘플을 메모리에서 활용할 준비를 마쳤습니다. 파이팅!")
else:
    # 일반 데이터셋 (Dataset)입니다.
    sampled_dataset = dataset.select(range(min(SAMPLE_COUNT, len(dataset))))
    print(f"\n[STEP 2] ♻️ 샘플 데이터셋 준비 (일반 로드)")


# ---------------------------------------------------------------------
# 🧠 초급 실습 1: 메타데이터 탐색 - "AI가 왜 필요한가?" 이해하기
# ---------------------------------------------------------------------
print("\n" + "=" * 70)
print("🔬 [실습 1] 메타데이터 탐색: '구름'과 '계절' 분석하기")
print("==================================================================")
print("튜터: AI 모델은 데이터가 얼마나 '특정 패턴'을 가지는지 알아야 똑똑해지죠! 우리는 이메타데이터를 분석해서 '구름 제거'가 필요했던 상황을 찾아볼 거예요.")

# 1. 분석할 샘플을 임시 리스트에 저장합니다.
analysis_list = []
print("   -> 샘플 데이터셋을 순회하며 필요한 정보만 추출합니다...")

for sample in sampled_dataset:
    # 안전하게 'features' 내부의 값에 접근합니다.
    features = sample.get('features', {})
    
    # 'cloudy'와 'season' 값이 존재하는지 확인합니다.
    if 'cloudy' in features and 'season' in features:
        # 'cloudy'가 1 (True)이고, 'season'이 'spring'인 경우만 추출합니다. (가장 흥미로운 경우)
        if features['cloudy'] == 1 and features['season'] == 'spring':
             analysis_list.append(sample)

print(f"   🎉 분석 완료! {len(analysis_list)}개의 '봄에 구름이 많은' 흥미로운 샘플을 찾아냈습니다.")
print("   (이 샘플들은 AI의 '구름 제거' 작업이 필요했을 확률이 매우 높았어요!)")

# ---------------------------------------------------------------------
# 🚀 초급 실습 2: 조건 필터링 및 패턴 검색 (AI의 입력 전처리 단계)
# ---------------------------------------------------------------------
print("\n" + "=" * 70)
print("🔎 [실습 2] 조건 필터링: 'SAR 데이터'가 중요한 장면만 골라보기")
print("==================================================================")
print("튜터: 현실의 데이터는 너무 많아서, 우리는 AI에게 '이 조건에 맞는 데이터만 봐!'라고 알려줘야 합니다. 이게 바로 데이터 전처리(Preprocessing)예요.")

# 1. 필터링할 리스트를 만듭니다.
filtered_scenes = []
print("   -> 'SAR' 특징이 있고, 동시에 'target'이 존재하는 장면을 찾습니다...")

for sample in sampled_dataset:
    features = sample.get('features', {})
    
    # 조건 1: SAR 데이터가 존재한다 (원격 탐사에서 중요한 센서 데이터)
    sar_check = features.get('sar')
    
    # 조건 2: 'target' (목표) 값이 설정되어 있다.
    target_check = features.get('target')
    
    if sar_check == 1 and target_check == 1:
        # 이 샘플은 SAR 데이터와 목표 데이터가 모두 준비된, 매우 중요한 샘플입니다!
        filtered_scenes.append(sample)

print(f"   ✨ 필터링 성공! 총 {len(filtered_scenes)}개의 고가치(High-Value) 샘플을 찾았습니다.")
print("   (이 샘플들로만 AI 모델을 훈련하면, '구름 제거'의 정확도를 훨씬 높일 수 있을 거예요!)")


# ---------------------------------------------------------------------
# 📈 초급 실습 3: 간단한 시계열 분석 시뮬레이션 (Season 추이 확인)
# ---------------------------------------------------------------------
print("\n" + "=" * 70)
print("📊 [실습 3] 계절별 데이터 빈도 분석 시뮬레이션")
print("==================================================================")
print("튜터: 데이터를 보기 좋게 분류하는 것도 AI 프로젝트의 핵심이에요. 어떤 계절에 어떤 일이 많이 발생했는지 분석해 봅시다.")

season_counter = {}

for sample in sampled_dataset:
    features = sample.get('features', {})
    season = features.get('season')
    
    if season and season not in season_counter:
        season_counter[season] = 0
    
    season_counter[season] += 1

# 결과 출력
print("\n✨ 계절별 샘플 개수 분석 결과:")
for season, count in sorted(season_counter.items(), key=lambda item: item[0]):
    print(f"  🍂 {season.upper()} 시즌: 총 {count}개 샘플 발견")

print("\n" + "=" * 70)
print("🎓 튜토리얼 완료! 🎉")
print("축하합니다! 이제 당신은 위성 데이터를 단순한 이미지로 보는 것을 넘어,")
print("메타데이터(Season, Cloudy, SAR)를 활용해 AI가 어떤 상황에서 필요한지 깊이 있게 분석할 수 있게 되었습니다!")
print("이것이 바로 'AI의 눈'을 갖추는 첫 단계랍니다! 💪")